<a href="https://colab.research.google.com/github/nsisongSunday7778/Toxic-comment-demo/blob/main/NLP_project_final_toxic_Comment_BY_NSISONG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Imports & NLTK Downloads

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns, re, nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (classification_report, confusion_matrix, accuracy_score,
                              precision_score, recall_score, f1_score, roc_auc_score,
                              precision_recall_curve)
from imblearn.over_sampling import SMOTE

nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


True

**Result:** `True` — confirms NLTK's stopword list and lemma dictionary downloaded successfully. This just loads every tool the notebook needs; nothing is computed yet.

## 2. Load the Training Data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

train_text = pd.read_json('/content/drive/MyDrive/Training_data.json', lines=True)
train_text.head()

Mounted at /content/drive


,text,parent_comment,article_title,article_url,platform,platform_id,composite_toxic
0,"WTF, y'all never made MRE fart balloons in the...",None,Triangular UFO hovers over California military...,https://www.dailymail.co.uk/news/article-12112...,reddit,jlcm021,"[[False, 74], [True, 323], [False, 1028], [Fal..."
1,No apologies !! McCall has balls ! Ccp is not...,None,China sentences elderly US citizen to life in ...,https://www.cnn.com/2023/05/15/china/china-jai...,youtube,Ugws8gNW7eJyE9VHeM14AaABAg,"[[False, 216], [False, 197], [False, 1039], [F..."
2,What ever you need to tell yourself to sleep a...,I wonder how many undercover agents will be go...,Jan. 6 defendant who put foot on desk in Pelos...,https://www.cbsnews.com/news/richard-barnett-j...,youtube,UgxHlqwNcVssLHUr4yF4AaABAg.9q7kOunSlu-9q7lHH4he6S,"[[True, 192], [True, 193], [True, 260], [True,..."
3,@exZACKly @CBSNews Fuck off Nazi,@NCmylo @CBSNews Lol. Stop choosing to be an ...,19-year-old Missouri man arrested in U-Haul cr...,https://www.cbsnews.com/news/u-haul-crash-lafa...,twitter,1661025155047637000,"[[True, 92], [False, 218], [True, 69], [True, ..."
4,Texas is a republican sponsored killing ground...,None,At Least 8 Killed After Driver Plows Car Into ...,https://www.nytimes.com/2023/05/07/us/car-pede...,youtube,UgwpAfn9RIV0cHfhp4R4AaABAg,"[[False, 56], [True, 207], [False, 218], [Fals..."


**Result:** A table with 7 columns — `text`, `parent_comment`, `article_title`, `article_url`, `platform`, `platform_id`, `composite_toxic`. Only `text` (the comment) and `composite_toxic` (raw per-annotator votes) matter for this project; the rest is just metadata about where each comment came from.

## 3. Quick EDA

In [ ]:
print('shape of the data:\n', train_text.shape)
print('\nInformation about the data:')
train_text.info()

# EDA: check for missing values
train_text.isna().sum()

shape of the data:
 (4000, 7)

Information about the data:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4000 entries, 0 to 3999
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   text             4000 non-null   object
 1   parent_comment   1402 non-null   object
 2   article_title    4000 non-null   object
 3   article_url      4000 non-null   object
 4   platform         4000 non-null   object
 5   platform_id      4000 non-null   object
 6   composite_toxic  4000 non-null   object
dtypes: object(7)
memory usage: 218.9+ KB


,0
text,0
parent_comment,2598
article_title,0
article_url,0
platform,0
platform_id,0
composite_toxic,0


**Result:** 4,000 rows, 7 columns. `parent_comment` has 2,598 missing values (many comments aren't replies to anything, so they simply have no parent). `text` and `composite_toxic` — the two columns this project actually uses — have **zero** missing values, so no cleanup is needed there.

## 4. Keep Only What's Needed, Rename the Label Column

In [ ]:
train_text = train_text[['text', 'composite_toxic']]
train_text = train_text.rename(columns={'composite_toxic': 'label'})
train_text.head()

,text,label
0,"WTF, y'all never made MRE fart balloons in the...","[[False, 74], [True, 323], [False, 1028], [Fal..."
1,No apologies !! McCall has balls ! Ccp is not...,"[[False, 216], [False, 197], [False, 1039], [F..."
2,What ever you need to tell yourself to sleep a...,"[[True, 192], [True, 193], [True, 260], [True,..."
3,@exZACKly @CBSNews Fuck off Nazi,"[[True, 92], [False, 218], [True, 69], [True, ..."
4,Texas is a republican sponsored killing ground...,"[[False, 56], [True, 207], [False, 218], [Fals..."


**Result:** Down to two columns: `text` and `label`. At this point `label` is still a list like `[[False, 74], [True, 323], ...]` — one `[is_toxic, worker_id]` pair per human annotator — not yet a single answer.

## 5. Build a Single Binary Label via Majority Vote

In [ ]:
def flatten_annotations(row):
    if not isinstance(row, (list, tuple)) or len(row) == 0:
        return False
    annotations = [label for label, worker_id in row]
    return sum(annotations) > (len(annotations) / 2)

train_text['label'] = train_text['label'].apply(flatten_annotations)

**What it does:** for each comment, pulls out just the True/False votes (ignoring worker IDs) and returns `True` only if more than half the annotators called it toxic.

**Result:** `label` is now a clean single `True`/`False` value per comment — a proper classification target.

In [ ]:
train_text['label'].value_counts()
# False    2974
# True     1026

train_text['label'].value_counts(normalize=True)
# False    0.7435
# True     0.2565

,proportion
label,
False,0.7435
True,0.2565


**Result:** ~74% of comments are non-toxic, ~26% are toxic — roughly a 3-to-1 imbalance. **This single number is the reason the rest of the notebook exists.** A model that always guesses "not toxic" would already score 74% accuracy while catching zero real toxic comments — so accuracy alone can't be trusted.

## 6. Text Cleaning (Lemmatization)

In [ ]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))  # computed once, not per word

def lamming(content):
    lemma_content = re.sub('[^a-zA-Z]', ' ', str(content))
    lemma_content = lemma_content.lower()
    lemma_content = lemma_content.split()
    lemma_content = [lemmatizer.lemmatize(word) for word in lemma_content if word not in stop_words]
    lemma_content = ' '.join(lemma_content)
    return lemma_content

train_text['text_lemma_processed'] = train_text['text'].apply(lamming)
train_text.head()

,text,label,text_lemma_processed
0,"WTF, y'all never made MRE fart balloons in the...",False,wtf never made mre fart balloon stump fucking ...
1,No apologies !! McCall has balls ! Ccp is not...,False,apology mccall ball ccp nothing full crap
2,What ever you need to tell yourself to sleep a...,True,ever need tell sleep night fucking retard
3,@exZACKly @CBSNews Fuck off Nazi,True,exzackly cbsnews fuck nazi
4,Texas is a republican sponsored killing ground...,False,texas republican sponsored killing ground teac...


**What it does:** strips punctuation/numbers, lowercases everything, removes common filler words ("the", "is", "and"...), and reduces each remaining word to its dictionary root (e.g. "running" → "run").

**Result (example):** `"WTF, y'all never made MRE fart balloons..."` becomes `"wtf never made mre fart balloon..."` — shorter, normalized text that's easier for a model to find patterns in. This cleaned column, not the raw `text`, is what gets vectorized next.

## 7. Split Before Vectorizing

In [ ]:
X = train_text['text_lemma_processed']
y = train_text['label']

le = LabelEncoder()
y_encoded = le.fit_transform(y)  # False/True -> 0/1

X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

print('Train size:', X_train.shape[0], '| Test size:', X_test.shape[0])
print('Train class balance:', pd.Series(y_train).value_counts(normalize=True).round(3).to_dict())
print('Test class balance: ', pd.Series(y_test).value_counts(normalize=True).round(3).to_dict())

Train size: 3200 | Test size: 800
Train class balance: {0: 0.743, 1: 0.257}
Test class balance:  {0: 0.744, 1: 0.256}


**Result:** 3,200 training comments / 800 test comments. `stratify=y_encoded` keeps the same ~74/26 split in both halves (confirmed: train is 74.3%/25.7%, test is 74.4%/25.6%). Splitting **before** vectorizing and before any resampling matters — it stops information from the test set leaking into training.

## 8. Count-Vectorize the Text

In [ ]:
cvt = CountVectorizer()
X_train_transform = cvt.fit_transform(X_train)   # learn vocabulary from training only
X_test_transform = cvt.transform(X_test)         # reuse that same vocabulary

print('X_train_transform shape:', X_train_transform.shape)
print('X_test_transform shape: ', X_test_transform.shape)
print('Vocabulary size:', len(cvt.vocabulary_))

X_train_transform shape: (3200, 10647)
X_test_transform shape:  (800, 10647)
Vocabulary size: 10647


**Result:** Training matrix shape `(3200, 10647)`, test matrix shape `(800, 10647)` — 10,647 unique words found in the training comments become 10,647 numeric columns. Each comment becomes a row of word counts. Fitting only on training data (never on test) is what keeps this a fair, leakage-free evaluation.

## 9. Experiment 1 — Baseline (No Imbalance Handling)

Three models trained as-is on the imbalanced 74/26 data, to serve as a control group.

### 9a. Logistic Regression (baseline)

In [ ]:
lrmodel1 = LogisticRegression(max_iter=1000, random_state=42)
lrmodel1.fit(X_train_transform, y_train)

X_test_predict_lr = lrmodel1.predict(X_test_transform)
print(classification_report(y_test, X_test_predict_lr))
print(confusion_matrix(y_test, X_test_predict_lr))

              precision    recall  f1-score   support

           0       0.79      0.93      0.86       595
           1       0.59      0.28      0.38       205

    accuracy                           0.77       800
   macro avg       0.69      0.61      0.62       800
weighted avg       0.74      0.77      0.73       800

[[556  39]
 [148  57]]


**Result:** Accuracy 0.766, Precision (Toxic) 0.594, Recall (Toxic) 0.278, F1 (Toxic) 0.379.

### 9b. Naive Bayes (baseline)

In [ ]:
nbmodel1 = MultinomialNB()
nbmodel1.fit(X_train_transform, y_train)

X_test_predict_nb = nbmodel1.predict(X_test_transform)
print(classification_report(y_test, X_test_predict_nb))
print(confusion_matrix(y_test, X_test_predict_nb))

              precision    recall  f1-score   support

           0       0.80      0.91      0.85       595
           1       0.55      0.33      0.41       205

    accuracy                           0.76       800
   macro avg       0.68      0.62      0.63       800
weighted avg       0.73      0.76      0.74       800

[[541  54]
 [138  67]]


**Result:** Accuracy 0.760, Precision (Toxic) 0.554, Recall (Toxic) 0.327, F1 (Toxic) 0.411.

### 9c. Random Forest (baseline)

In [ ]:
rfmodel1 = RandomForestClassifier(random_state=42, n_jobs=-1)
rfmodel1.fit(X_train_transform, y_train)

X_test_predict_rf = rfmodel1.predict(X_test_transform)
print(classification_report(y_test, X_test_predict_rf))
print(confusion_matrix(y_test, X_test_predict_rf))

              precision    recall  f1-score   support

           0       0.78      0.95      0.86       595
           1       0.62      0.23      0.33       205

    accuracy                           0.77       800
   macro avg       0.70      0.59      0.60       800
weighted avg       0.74      0.77      0.72       800

[[566  29]
 [158  47]]


**Result:** Accuracy 0.766, Precision (Toxic) 0.618, Recall (Toxic) 0.229, F1 (Toxic) 0.335.

**Reading Experiment 1 overall:** all three land around 76–77% accuracy — suspiciously close to the 74% you'd get by always guessing "not toxic." Recall on the toxic class is the tell: all three models catch fewer than 1 in 3 real toxic comments (Random Forest catches only 22.9%). High accuracy, poor toxic-catching — exactly the imbalance trap flagged in Section 5.

## 10. Experiment 2 — Class-Weight Balancing

Instead of changing the data, the loss function is reweighted so mistakes on toxic comments cost more.

### 10a. Logistic Regression (`class_weight='balanced'`)

In [ ]:
logmodel2 = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
logmodel2.fit(X_train_transform, y_train)

X_test_prediction = logmodel2.predict(X_test_transform)
print(classification_report(y_test, X_test_prediction))
print(confusion_matrix(y_test, X_test_prediction))

              precision    recall  f1-score   support

           0       0.83      0.84      0.83       595
           1       0.52      0.50      0.51       205

    accuracy                           0.75       800
   macro avg       0.67      0.67      0.67       800
weighted avg       0.75      0.75      0.75       800

[[499  96]
 [103 102]]


**Result:** Accuracy 0.751, Precision (Toxic) 0.515, Recall (Toxic) 0.498, F1 (Toxic) 0.506.

### 10b. Naive Bayes (sample-weighted)

In [ ]:
# MultinomialNB has no class_weight parameter, so we compute per-sample weights and pass them into .fit() instead.
sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

nbmodel2 = MultinomialNB()
nbmodel2.fit(X_train_transform, y_train, sample_weight=sample_weights)

X_test_predict_nb2 = nbmodel2.predict(X_test_transform)
print(classification_report(y_test, X_test_predict_nb2))
print(confusion_matrix(y_test, X_test_predict_nb2))

              precision    recall  f1-score   support

           0       0.86      0.64      0.74       595
           1       0.40      0.70      0.51       205

    accuracy                           0.66       800
   macro avg       0.63      0.67      0.62       800
weighted avg       0.74      0.66      0.68       800

[[383 212]
 [ 62 143]]


**Result:** Accuracy 0.658, Precision (Toxic) 0.403, Recall (Toxic) 0.698, F1 (Toxic) 0.511.

### 10c. Random Forest (`class_weight='balanced'`)

In [ ]:
rfmodel2 = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_split=20,
    min_samples_leaf=10,
    random_state=42,
    n_jobs=-1,
    class_weight='balanced'
)
rfmodel2.fit(X_train_transform, y_train)

X_test_prediction = rfmodel2.predict(X_test_transform)
print(classification_report(y_test, X_test_prediction))
print(confusion_matrix(y_test, X_test_prediction))

              precision    recall  f1-score   support

           0       0.84      0.76      0.80       595
           1       0.46      0.60      0.52       205

    accuracy                           0.71       800
   macro avg       0.65      0.68      0.66       800
weighted avg       0.75      0.71      0.73       800

[[450 145]
 [ 83 122]]


**Result:** Accuracy 0.715, Precision (Toxic) 0.457, Recall (Toxic) 0.595, F1 (Toxic) 0.517.

**Reading Experiment 2 overall:** every model's recall roughly doubles versus the baseline — Naive Bayes now catches 69.8% of toxic comments (up from 32.7%). The trade-off is visible in accuracy, which drops for all three (most sharply for Naive Bayes, to 65.8%), because each model now flags more non-toxic comments as toxic too. Random Forest has the best F1 of the three here (0.517) — the best balance between catching toxic comments and not over-flagging non-toxic ones of all nine models trained in this notebook.

## 11. Experiment 3 — SMOTE Oversampling

SMOTE synthesizes new minority-class (toxic) samples by interpolating between existing ones, so the training set becomes balanced 50/50. **Critical rule:** SMOTE is fit only on the vectorized *training* data (`X_train_transform`, `y_train`) — `X_test_transform` is never touched by it. Evaluating on synthetically-balanced test data would give an unrealistically optimistic score that doesn't reflect real-world (imbalanced) traffic.

In [ ]:
print('Before SMOTE:', pd.Series(y_train).value_counts().to_dict())

smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_transform, y_train)

print('After SMOTE: ', pd.Series(y_train_smote).value_counts().to_dict())

Before SMOTE: {0: 2379, 1: 821}
After SMOTE:  {0: 2379, 1: 2379}


**Result:** Before SMOTE: `{0: 2379, 1: 821}`. After SMOTE: `{0: 2379, 1: 2379}` — a perfect 50/50 training set, achieved entirely with synthetic toxic-class examples.

### 11a. Naive Bayes (SMOTE)

In [ ]:
nbmodel3 = MultinomialNB()
nbmodel3.fit(X_train_smote, y_train_smote)

X_test_prediction = nbmodel3.predict(X_test_transform)
print(classification_report(y_test, X_test_prediction))
print(confusion_matrix(y_test, X_test_prediction))

              precision    recall  f1-score   support

           0       0.84      0.77      0.80       595
           1       0.46      0.57      0.51       205

    accuracy                           0.72       800
   macro avg       0.65      0.67      0.66       800
weighted avg       0.74      0.72      0.73       800

[[460 135]
 [ 89 116]]


**Result:** Accuracy 0.720, Precision (Toxic) 0.462, Recall (Toxic) 0.566, F1 (Toxic) 0.509.

### 11b. Logistic Regression (SMOTE)

In [ ]:
logmodel3 = LogisticRegression(max_iter=1000, random_state=42)
logmodel3.fit(X_train_smote, y_train_smote)

X_test_prediction = logmodel3.predict(X_test_transform)
print(classification_report(y_test, X_test_prediction))
print(confusion_matrix(y_test, X_test_prediction))

              precision    recall  f1-score   support

           0       0.82      0.62      0.71       595
           1       0.36      0.61      0.45       205

    accuracy                           0.62       800
   macro avg       0.59      0.62      0.58       800
weighted avg       0.71      0.62      0.64       800

[[371 224]
 [ 79 126]]


**Result:** Accuracy 0.621, Precision (Toxic) 0.360, Recall (Toxic) 0.615, F1 (Toxic) 0.454.

### 11c. Random Forest (SMOTE)

In [ ]:
rfmodel3 = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_split=20,
    min_samples_leaf=10,
    random_state=42,
    n_jobs=-1
    # class_weight intentionally omitted: combining SMOTE with class_weight would double-correct for imbalance
)
rfmodel3.fit(X_train_smote, y_train_smote)

X_test_prediction = rfmodel3.predict(X_test_transform)
print(classification_report(y_test, X_test_prediction))
print(confusion_matrix(y_test, X_test_prediction))

              precision    recall  f1-score   support

           0       0.81      0.53      0.64       595
           1       0.32      0.65      0.43       205

    accuracy                           0.56       800
   macro avg       0.57      0.59      0.53       800
weighted avg       0.69      0.56      0.59       800

[[313 282]
 [ 72 133]]


**Result:** Accuracy 0.558, Precision (Toxic) 0.320, Recall (Toxic) 0.649, F1 (Toxic) 0.429.

**Reading Experiment 3 overall:** recall rises again for Logistic Regression and Random Forest (61.5% and 64.9%), but precision and overall accuracy fall further — Random Forest's accuracy drops all the way to 55.8%, meaning it's now wrong on almost half the test set, mostly by over-flagging non-toxic comments as toxic.

## 12. Full Comparison Table (All 9 Models)

In [ ]:
def evaluate_model(model, X, y_true, name):
    y_pred = model.predict(X)
    return {
        'Model': name,
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision (Toxic)': precision_score(y_true, y_pred),
        'Recall (Toxic)': recall_score(y_true, y_pred),
        'F1 (Toxic)': f1_score(y_true, y_pred),
        'F1 (Macro)': f1_score(y_true, y_pred, average='macro'),
    }

rows = [
    evaluate_model(lrmodel1, X_test_transform, y_test, 'Logistic Regression - Baseline'),
    evaluate_model(nbmodel1, X_test_transform, y_test, 'Naive Bayes - Baseline'),
    evaluate_model(rfmodel1, X_test_transform, y_test, 'Random Forest - Baseline'),
    evaluate_model(logmodel2, X_test_transform, y_test, 'Logistic Regression - Class Weight'),
    evaluate_model(nbmodel2, X_test_transform, y_test, 'Naive Bayes - Class Weight'),
    evaluate_model(rfmodel2, X_test_transform, y_test, 'Random Forest - Class Weight'),
    evaluate_model(logmodel3, X_test_transform, y_test, 'Logistic Regression - SMOTE'),
    evaluate_model(nbmodel3, X_test_transform, y_test, 'Naive Bayes - SMOTE'),
    evaluate_model(rfmodel3, X_test_transform, y_test, 'Random Forest - SMOTE'),
]

comparison_df = pd.DataFrame(rows).round(3)
comparison_df

,Model,Accuracy,Precision (Toxic),Recall (Toxic),F1 (Toxic),F1 (Macro)
0,Logistic Regression - Baseline,0.766,0.594,0.278,0.379,0.617
1,Naive Bayes - Baseline,0.760,0.554,0.327,0.411,0.630
2,Random Forest - Baseline,0.766,0.618,0.229,0.335,0.596
3,Logistic Regression - Class Weight,0.751,0.515,0.498,0.506,0.670
4,Naive Bayes - Class Weight,0.658,0.403,0.698,0.511,0.624
5,Random Forest - Class Weight,0.715,0.457,0.595,0.517,0.657
6,Logistic Regression - SMOTE,0.621,0.360,0.615,0.454,0.582
7,Naive Bayes - SMOTE,0.720,0.462,0.566,0.509,0.656
8,Random Forest - SMOTE,0.558,0.320,0.649,0.429,0.534


**What it does:** assembles every model above into one sortable table. It's the single most useful output in the notebook — every design choice (baseline vs. class-weight vs. SMOTE, and which algorithm) can be compared side-by-side on Accuracy, Precision, Recall, and F1 for the toxic class.

**Result summary:**

| Model | Strategy | Accuracy | Precision | Recall | F1 |
|---|---|---|---|---|---|
| Logistic Regression | Baseline | 0.766 | 0.594 | 0.278 | 0.379 |
| Naive Bayes | Baseline | 0.760 | 0.554 | 0.327 | 0.411 |
| Random Forest | Baseline | 0.766 | 0.618 | 0.229 | 0.335 |
| Logistic Regression | Class-Weight | 0.751 | 0.515 | 0.498 | 0.506 |
| Naive Bayes | Class-Weight | 0.658 | 0.403 | **0.698** | 0.511 |
| Random Forest | Class-Weight | 0.715 | 0.457 | 0.595 | **0.517** |
| Logistic Regression | SMOTE | 0.621 | 0.360 | 0.615 | 0.454 |
| Naive Bayes | SMOTE | 0.720 | 0.462 | 0.566 | 0.509 |
| Random Forest | SMOTE | 0.558 | 0.320 | 0.649 | 0.429 |

**Best pure toxic-comment catcher (recall): Naive Bayes – Class Weight (69.8%).** **Best balance of catching toxic comments vs. false alarms (F1): Random Forest – Class Weight (0.517).**

In [ ]:
import pandas as pd

# Load unlabeled test data
test_text = pd.read_json(
    "/content/drive/MyDrive/Test_data.json",
    lines=True
)

In [ ]:
test_text.head()

,text,parent_comment,article_title,article_url,platform,platform_id
0,Ukrainian Bullshit.,Russian Propaganda,Kremlin drone: Zelensky denies Ukraine attacke...,https://www.bbc.com/news/world-europe-65471904,youtube,UgxjV6HRpnD6FUmw8aV4AaABAg.9pH-CgX5yEH9pH7BMIfAz5
1,@LibDems No one likes you.\nYou denied democra...,None,"UK economy shrank 0.3% in March, ONS figures show",https://news.sky.com/story/uk-economy-shrank-0...,twitter,1657052099564150784
2,@EPurpera @BBCNews POS terrorist dictator Putr...,@BBCNews They should make peace talk.,Ukraine war: Kyiv hit by new massive Russian d...,https://www.bbc.com/news/world-65736730,twitter,1662672469205958656
3,@howardfineman @darkblue714 Bullshit. CNN set ...,None,Opinion | Why CNN's Trump town hall was always...,https://www.msnbc.com/opinion/msnbc-opinion/cn...,twitter,1656508255454019587
4,"The war will be won by who ""wins"" the race bet...",What is the pope gonna do? Pray and throw a co...,Zelenskyy to meet with Pope Francis at Vatican...,https://apnews.com/article/zelenskyy-italy-vis...,reddit,jk1pm1m


In [ ]:
test_data=test_text['text']

In [ ]:
test_data.shape

(500,)

In [ ]:
test_data_clean=test_data.apply(lamming)

In [ ]:
test_data_clean.head()

,text
0,ukrainian bullshit
1,libdems one like denied democracy actively wor...
2,epurpera bbcnews po terrorist dictator putrid ...
3,howardfineman darkblue bullshit cnn set fail k...
4,war win race russian collapse even ukraine tak...


In [ ]:
test_train=cvt.transform(test_data_clean)

In [ ]:
test_train[0:10].toarray()

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]])

In [ ]:
logmodel_prediction=logmodel2.predict(test_train)
nbmodel_prediction=nbmodel2.predict(test_train)
rfmodel_prediction=rfmodel2.predict(test_train)

In [ ]:
predicted_results=pd.DataFrame({'text': test_data_clean,
  'logmodel_prediction':logmodel_prediction,
  'nbmodel_prediction':nbmodel_prediction,
  'rfmodel_prediction':rfmodel_prediction
})

In [ ]:
predicted_results.head(50)

,text,logmodel_prediction,nbmodel_prediction,rfmodel_prediction
0,ukrainian bullshit,1,1,1
1,libdems one like denied democracy actively wor...,1,1,0
2,epurpera bbcnews po terrorist dictator putrid ...,1,1,1
3,howardfineman darkblue bullshit cnn set fail k...,0,0,1
4,war win race russian collapse even ukraine tak...,0,0,0
5,say fuck lewis powell jr even,1,1,1
6,stand fascist modi support russia,1,1,1
7,lol many u two three time one ever seems want ...,0,0,0
8,phil lewis damn white supremacist,0,1,1
9,thought prayer next one city near,0,0,0


In [ ]:
predicted_results['toxic and non toxic prediction rf']=predicted_results["rfmodel_prediction"].astype(bool)

In [ ]:
predicted_results

,text,logmodel_prediction,nbmodel_prediction,rfmodel_prediction,toxic and non toxic prediction rf
0,ukrainian bullshit,1,1,1,True
1,libdems one like denied democracy actively wor...,1,1,0,False
2,epurpera bbcnews po terrorist dictator putrid ...,1,1,1,True
3,howardfineman darkblue bullshit cnn set fail k...,0,0,1,True
4,war win race russian collapse even ukraine tak...,0,0,0,False
...,...,...,...,...,...
495,without accountability killing displacing whol...,0,0,0,False
496,shlomoben know anyone oh right hyperbole lie,0,0,0,False
497,cnn would ct claim anything grand nephew commi...,0,1,0,False
498,rely friend winnie lmao okay comrade dont make...,1,1,0,False


## 13. Deploying the Final Model — Live Comment Check

Using **Random Forest (Class-Weight)** as the final model — it had the best F1 balance (0.517) in the comparison table above. Type in any comment and the model returns True (toxic) or False (not toxic).

**Note:** this is a live prediction on a brand-new sentence, not a scored evaluation — there's no ground-truth label for a comment someone just typed, so no accuracy/precision/recall applies here, only the model's guess.

In [ ]:
def predict_toxicity(comment, model=rfmodel2, vectorizer=cvt):
    cleaned = lamming(comment)                # same cleaning used in training
    vectorized = vectorizer.transform([cleaned])  # reuse the fitted CountVectorizer, NOT fit_transform
    prediction = model.predict(vectorized)[0]
    return bool(prediction)                     # 1 -> True (toxic), 0 -> False (not toxic)

# Type a comment in and get True/False back
user_comment = input("Enter a comment: ")
print("Toxic:", predict_toxicity(user_comment))

Enter a comment: I dislike you
Toxic: False


## 14. Saving the Model for Deployment (.pkl)

For a demo, you need **two** files, not one — the trained model AND the fitted vectorizer. Without the exact same vectorizer, new text can't be turned into the right numeric columns, and predictions will be garbage or error out.

In [ ]:
import pickle
pickle.dump(rfmodel2,open('toxic_rf_model2','wb'))

In [ ]:
import joblib

joblib.dump(rfmodel2, '/content/drive/MyDrive/toxic_rf_model.pkl')
joblib.dump(cvt, '/content/drive/MyDrive/toxic_vectorizer.pkl')

print('Saved model and vectorizer to Google Drive.')

Saved model and vectorizer to Google Drive.


**In your demo app**, load both back and reuse the same `predict_toxicity` logic:
```python
import joblib

rfmodel2 = joblib.load('toxic_rf_model.pkl')
cvt = joblib.load('toxic_vectorizer.pkl')

def predict_toxicity(comment, model=rfmodel2, vectorizer=cvt):
    cleaned = lamming(comment)
    vectorized = vectorizer.transform([cleaned])
    prediction = model.predict(vectorized)[0]
    return bool(prediction)
```
You'll also need to bring the `lamming` function (and `stopwords`/`lemmatizer` it depends on) into whatever script runs the demo — pickling the model doesn't carry your cleaning function along with it.

In [ ]:
import joblib

rfmodel2 = joblib.load('/content/drive/MyDrive/toxic_rf_model.pkl')
cvt = joblib.load('/content/drive/MyDrive/toxic_vectorizer.pkl')

def predict_toxicity(comment, model=rfmodel2, vectorizer=cvt):
    cleaned = lamming(comment)
    vectorized = vectorizer.transform([cleaned])
    prediction = model.predict(vectorized)[0]
    return bool(prediction)

In [ ]:
new_comment = "This is a test comment, I hope it's not toxic."
prediction = predict_toxicity(new_comment)
print(f"The comment: '{new_comment}' is toxic: {prediction}")

new_comment_toxic = "You are a terrible person!"
prediction_toxic = predict_toxicity(new_comment_toxic)
print(f"The comment: '{new_comment_toxic}' is toxic: {prediction_toxic}")

The comment: 'This is a test comment, I hope it's not toxic.' is toxic: False
The comment: 'You are a terrible person!' is toxic: False
